# 第7章　医療画像の基礎 ― DICOM、NIfTI、前処理**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## CTの画像は、どんな数字でできているのか ― 代表的な値を押さえる

In [ ]:
import pydicomds = pydicom.dcmread("slice_001.dcm")print(ds.Rows, ds.Columns)                    # 512 512print(ds.PixelSpacing)                        # [0.625, 0.625]     ← 面内 [mm]print(ds.SliceThickness)                      # 5.0                ← 厚み [mm]print(getattr(ds, "SpacingBetweenSlices", None))   # 無い装置もあるので getattr で守る# AIに渡す前に、必ずこの3つを確認する。# 「面内 0.625mm、スライス間隔 5mm」なら、ボクセル間隔は 0.625 x 0.625 x 5.0 mm。# 間隔は SliceThickness（厚み）ではなく、隣り合うスライスの ImagePositionPatient の# 差から求める（厚みと間隔の違いは第8章）。# この非等方性を無視したまま3次元学習を回すと、体積も距離も歪んだまま学ぶことになる。

In [ ]:
import pydicomimport numpy as npds = pydicom.dcmread("image.dcm")pixel_array = ds.pixel_array        # 画素値の配列を取り出す# CTの場合、生の画素値をCT値（HU）に変換する（タグ欠落は例外にする）# タグが無いときに 1/0 で埋めてはいけない。格納値をHUと称して処理してしまう。if "RescaleSlope" not in ds or "RescaleIntercept" not in ds:    raise ValueError("HUへの変換係数が無い。この経路では変換不能として扱う")hu = pixel_array * float(ds.RescaleSlope) + float(ds.RescaleIntercept)# ※これは「単フレームCTで、Modality LUTが線形」という前提の最短経路。#   マルチフレームやLUTを持つ形式は、別の変換経路を確認してから扱う。

## 数字で追う ― 生の格納値からHUへ、そしてMONOCHROME1の落とし穴

In [ ]:
import pydicomds = pydicom.dcmread("xray.dcm")arr = ds.pixel_arrayif ds.PhotometricInterpretation == "MONOCHROME1":    # この式が成り立つのは「符号なし格納・LUTなし」の単純な場合だけ。前提を確かめる。    assert int(getattr(ds, "PixelRepresentation", 0)) == 0, "符号つき格納。別経路で扱う"    assert "ModalityLUTSequence" not in ds and "VOILUTSequence" not in ds, "LUTがある"    vmax = 2 ** int(ds.BitsStored) - 1   # 理論上の最大値。arr.max() だと画像ごとに基準が動く    arr = vmax - arr          # 明暗を反転し、MONOCHROME2 と同じ向きにそろえる

## 7.2　NIfTIと3次元データ

In [ ]:
import nibabel as nibvol = nib.load("case_0001.nii.gz")data = vol.get_fdata()          # 3次元配列として取り出すprint(data.shape)               # → (512, 512, 200) などspacing = vol.header.get_zooms()  # ボクセルの物理的な大きさ（mm）

## コラム：DICOM・NIfTI・PNG・npy をどう使い分けるか

In [ ]:
from PIL import ImageImage.fromarray(hu.astype("uint8")).save("ct.png")  # 危険：HUが0〜255に潰れる

## 7.3　データセットのディレクトリ設計

```texttumor_Dataset/├── train/│   ├── images/          # CTボリューム（.nii.gz）│   │   └── case_0001.nii.gz│   └── labels/          # 正解ラベル（.nii.gz）│       └── case_0001.nii.gz├── test/                # 評価用（trainと同じ構成）│   └── ...└── dataset.json         # データセットの要約とラベル定義```

## ウィンドウ処理を数字で追う ― worked example

In [ ]:
def window_normalize(hu, center, width):    lo, hi = center - width / 2, center + width / 2    return np.clip((hu - lo) / (hi - lo), 0.0, 1.0)

## 数字で追う ― 正規化統計のリークを実演する

In [ ]:
import numpy as nptrain = np.array([10., 12., 14., 16., 12., 14.])   # 学習症例の画素平均val   = np.array([98., 102.])                       # 検証症例（別装置で高輝度）# NG（リーク）：学習と検証を混ぜて統計を取るall_mean = np.concatenate([train, val]).mean()      # → 34.75all_std  = np.concatenate([train, val]).std()        # → 約 37.7# OK（正しい）：統計は学習データだけで決め、検証はそれを借りて使うtr_mean, tr_std = train.mean(), train.std()          # → 13.0, 約 1.9